In [2]:
import sys
sys.path.append("..")

from model.simulate import run_plan
from model.optimize import optimize_country

print(optimize_country("USA"))

[204, 204, 204, 204, 204, 204, 204, 204, 204, 289, 289, 289]


In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("../.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

llm.invoke("Say hello in five words.").content

'Hello! How are you today?'

In [4]:
import sys
sys.path.append("..")

import json
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from model.simulate import run_plan, salaries, staff, forecast, FIRE_COST
from model.optimize import optimize_country

load_dotenv("../.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done")




plan_df = pd.read_csv("../data/processed/plan.csv")

def country_facts(country):
    plan = plan_df[plan_df["country"] == country].sort_values("month")["agents"].tolist()
    best = run_plan(country, plan)
    keep = run_plan(country, [int(staff[country])] * 12)
    fc = forecast[forecast["country"] == country]
    salary = float(salaries[country])
    budget = float(fc["avg_eligible_budget"].iloc[0])
    net_best = best["revenue"].sum() - best["cost"].sum()
    net_keep = keep["revenue"].sum() - keep["cost"].sum()
    value_per_agent = 10 * 365 / 60 * 0.135 * budget

    return {
        "country": country,
        "agents_today": int(staff[country]),
        "recommended_jan_sep": int(plan[0]),
        "recommended_oct_dec": int(plan[9]),
        "agents_fired_in_january": int(best["fires"].sum()),
        "agents_hired_during_year": int(best["hires"].sum()),
        "one_time_firing_cost_usd": round(best["fires"].sum() * salary * (FIRE_COST + 1/12)),
        "net_value_recommended_usd_m": round(net_best / 1e6, 2),
        "net_value_if_unchanged_usd_m": round(net_keep / 1e6, 2),
        "net_value_gain_usd_m": round((net_best - net_keep) / 1e6, 2),
        "avg_daily_eligible_signups": round(float(fc["daily_eligible_base"].mean()), 1),
        "busiest_month": int(fc.loc[fc["daily_eligible_base"].idxmax(), "month"]),
        "max_avg_wait_days": round(float(best["avg_wait_days"].max()), 1),
        "advertisers_abandoned": round(float(best["abandoned"].sum())),
        "avg_utilization_pct": round(float(best["utilization"].mean() * 100)),
        "annual_salary_usd": round(salary),
        "avg_eligible_budget_usd": round(budget),
        "value_per_agent_vs_salary": round(value_per_agent / salary, 2),
    }

print(json.dumps(country_facts("Vietnam"), indent=2))

Setup done
{
  "country": "Vietnam",
  "agents_today": 59,
  "recommended_jan_sep": 77,
  "recommended_oct_dec": 107,
  "agents_fired_in_january": 0,
  "agents_hired_during_year": 48,
  "one_time_firing_cost_usd": 0,
  "net_value_recommended_usd_m": 9.78,
  "net_value_if_unchanged_usd_m": 3.2,
  "net_value_gain_usd_m": 6.57,
  "avg_daily_eligible_signups": 13.3,
  "busiest_month": 11,
  "max_avg_wait_days": 0.3,
  "advertisers_abandoned": 0,
  "avg_utilization_pct": 85,
  "annual_salary_usd": 17904,
  "avg_eligible_budget_usd": 17303,
  "value_per_agent_vs_salary": 7.94
}


In [5]:
from pydantic import BaseModel, Field

class MemoText(BaseModel):
    headline: str = Field(description="One sentence: the recommendation for this country.")
    reasons: list[str] = Field(description="Exactly 3 short reasons, each citing numbers from the facts.")
    risks: list[str] = Field(description="2 to 3 risks or caveats of the recommendation.")
    decision_needed: str = Field(description="One sentence: what the approver must decide or sign off on.")

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You write short staffing approval memos for Google Ads advertiser support. "
     "Use ONLY the numbers in the facts provided. Never calculate, estimate or invent numbers. "
     "Write for a busy manager: plain language, no jargon. "
     "Money is in US dollars; values ending in _usd_m are millions."),
    ("human",
     "Facts for {country} (from our 2026 staffing model):\n{facts}\n\n"
     "Background rules: each agent supports up to 10 advertisers for 60 days; "
     "new hires take 1 month to ramp up; firing costs 40% of annual salary plus 1 month notice; "
     "peak demand is in Q4."),
])

In [7]:
chain = prompt | llm.with_structured_output(MemoText)

facts = country_facts("Vietnam")
text = chain.invoke({"country": "Vietnam", "facts": json.dumps(facts, indent=2)})
text

MemoText(headline='Approve staffing increase for Vietnam to meet demand.', reasons=['Current staffing is 59 agents, while 77 are recommended for Jan-Sep.', 'Net value gain with recommended staffing is 6.57_m usd, compared to 3.2_m usd if unchanged.', 'Busiest month is November, indicating a need for increased support.'], risks=['New hires take 1 month to ramp up, which may delay support.', 'If demand exceeds projections, we may still face wait times.'], decision_needed='Please approve the staffing increase to 77 agents for Vietnam.')

In [8]:
def format_memo(facts, text):
    lines = [
        f"## {facts['country']}: {text.headline}", "",
        "| | Today | Recommended |",
        "|---|---|---|",
        f"| Agents Jan–Sep | {facts['agents_today']} | {facts['recommended_jan_sep']} |",
        f"| Agents Oct–Dec | {facts['agents_today']} | {facts['recommended_oct_dec']} |",
        f"| Net value 2026 | ${facts['net_value_if_unchanged_usd_m']}M | ${facts['net_value_recommended_usd_m']}M |",
        "", "**Why:**",
    ]
    lines += [f"- {r}" for r in text.reasons]
    lines += ["", "**Risks:**"]
    lines += [f"- {r}" for r in text.risks]
    lines += ["", f"**Decision needed:** {text.decision_needed}", ""]
    return "\n".join(lines)

print(format_memo(facts, text))

## Vietnam: Approve staffing increase for Vietnam to meet demand.

| | Today | Recommended |
|---|---|---|
| Agents Jan–Sep | 59 | 77 |
| Agents Oct–Dec | 59 | 107 |
| Net value 2026 | $3.2M | $9.78M |

**Why:**
- Current staffing is 59 agents, while 77 are recommended for Jan-Sep.
- Net value gain with recommended staffing is 6.57_m usd, compared to 3.2_m usd if unchanged.
- Busiest month is November, indicating a need for increased support.

**Risks:**
- New hires take 1 month to ramp up, which may delay support.
- If demand exceeds projections, we may still face wait times.

**Decision needed:** Please approve the staffing increase to 77 agents for Vietnam.



In [9]:
import os
os.makedirs("../report", exist_ok=True)

memos = ["# 2026 Staffing Approval Memos", ""]
for country in sorted(plan_df["country"].unique()):
    facts = country_facts(country)
    text = chain.invoke({"country": country, "facts": json.dumps(facts, indent=2)})
    memos.append(format_memo(facts, text))
    print("done:", country)

with open("../report/staffing_memos.md", "w", encoding="utf-8") as f:
    f.write("\n".join(memos))
print("Saved report/staffing_memos.md")

done: Argentina
done: Australia
done: Brazil
done: Canada
done: China
done: France
done: Germany
done: India
done: Indonesia
done: Italy
done: Japan
done: Mexico
done: Poland
done: South Africa
done: South Korea
done: Spain
done: Thailand
done: Turkey
done: UAE
done: UK
done: USA
done: Vietnam
Saved report/staffing_memos.md
